# Persistent project context for Claude

**Stop re-explaining your project every session — keep its essentials in one small structured file and pass it in the system prompt.**

Each new conversation, Claude starts cold: you re-describe the stack, the goals, how to run the tests. It's repetitive, and the grounding drifts from session to session.

A simple, durable fix: capture the project's essentials **once** in a small structured file, commit it to the repo, and inject it into the system prompt. Claude is grounded from the first message. This recipe shows the before/after with the raw Anthropic SDK — runnable in about 30 seconds.

## Setup

Just the Anthropic SDK. Your `ANTHROPIC_API_KEY` is read from the environment.

In [ ]:
%pip install -q anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-sonnet-4-6"  # swap for any current Claude model

def ask(system: str, question: str) -> str:
    msg = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=system,
        messages=[{"role": "user", "content": question}],
    )
    return msg.content[0].text

## Before — no context

Ask Claude a project-specific question with nothing to go on. It can only guess or ask you to share files.

In [3]:
print(ask(
    system="You are a coding assistant.",
    question="What database does this project use, and what is the test command?",
))

I don't see any project files, code, or repository content in our conversation. You haven't shared any files, configuration files, or code snippets that I could analyze.

To help you answer these questions, could you please share relevant files such as:

- **`package.json`** (for Node.js projects)
- **`requirements.txt`** or **`pyproject.toml`** (for Python projects)
- **`pom.xml`** or **`build.gradle`** (for Java projects)
- **`docker-compose.yml`**
- **`.env`** or config files
- **`README.md`**

Once you share those files, I can tell you what **database** the project uses and what the **test command** is.


## A structured context file

Capture what a newcomer (human or model) needs first: what the project is, the stack, and how to run and test it. Any readable format works — YAML is convenient and diffs well. Keep it **small**; it rides in every request.

In [4]:
project_context = """
project:
  name: orders-api
  goal: REST API for order management
  language: TypeScript
stack:
  backend: Express
  database: PostgreSQL
  testing: Jest
  runtime: Node.js
context:
  who: Backend developers
  what: Order CRUD with auth + webhooks
  why: Replace a legacy PHP service
  how: "npm test ; npm run dev"
"""

## After — grounded

Pass the context straight into the system prompt. Same question — now answered from ground truth.

In [5]:
grounded_system = (
    "You are a coding assistant. Below is the project's context. "
    "Treat it as ground truth.\n\n"
    + project_context
)

print(ask(
    system=grounded_system,
    question="What database does this project use, and what is the test command?",
))
# Expected: PostgreSQL; tests run with `npm test` (Jest). No guessing, no re-asking.

Based on the project context:

- **Database:** PostgreSQL
- **Test command:** `npm test`


## Define once, reuse everywhere

Commit the file to the repo so it travels with the code and updates in pull requests — context stays in sync with reality instead of living in someone's memory.

One structured source can also seed the per-tool files different assistants read (`CLAUDE.md`, `AGENTS.md`, `.cursorrules`, and so on): write it once, generate the views. The same block works for any model that accepts a system prompt.

## Going further

- **Keep it small and current.** Stale context misleads more than no context. Treat it like code — review it in PRs.
- **Generate it from the repo** rather than hand-writing, so it can't drift from `package.json`, the lockfile, or the test config.
- **Want a portable standard** instead of an ad-hoc file? There's an IANA-registered media type for exactly this — `.faf` (`application/vnd.faf+yaml`) — with generators that produce and score it (e.g. `claude-faf-mcp`, in the MCP Registry).